# P68 — Pilot P0: payload size vs corruption type and severity

**Project:** Benefit–Feasibility Coupling in Hierarchical Edge Inference (P68)
**Protocol:** `proposal_v4_final.md` (pilot protocol frozen 2026-10-01; SHA-256 `3d40c2d3…9b73`)
**Repo:** https://github.com/ArashSalehpourac/p68-hi-benefit-feasibility

What this notebook does (P0 only): decode source images → ImageNet-C-style corruption (severity 1–5) → **encode once** (JPEG q75, WebP q75) → record bytes → per-family check of the size direction against the pre-registered hypotheses (noise: size up; blur/contrast: size down).

What it does **not** do: it does not measure accuracy, the benefit G, queueing, or any gate other than the payload-direction precondition. Passing P0 is necessary, not sufficient.

**Data caveat.** ILSVRC validation images are already JPEGs, so the pipeline is *decode → corrupt → encode once* (not "raw"). `SOURCE="hf"` needs a Hugging Face token with the ImageNet-1k terms accepted. `SOURCE="imagenette"` is a public 10-class ImageNet subset: use it only for an indicative run, **not** as protocol evidence. `SOURCE="synthetic"` is for code smoke tests only. Images are never copied to GitHub; only CSVs, hashes and metadata.

## 1. Setup

In [ ]:
!pip -q install imagecorruptions datasets pandas scipy joblib 2>&1 | tail -1
import numpy as np
if not hasattr(np, "float_"):          # imagecorruptions (fog) still uses np.float_ under NumPy 2
    np.float_ = np.float64
import sys, os, io, json, time, hashlib, platform, random, subprocess, base64
import pandas as pd
from PIL import Image
from scipy.stats import spearmanr
from joblib import Parallel, delayed
from imagecorruptions import corrupt, get_corruption_names
IN_COLAB = "google.colab" in sys.modules
print("colab:", IN_COLAB, "| numpy", np.__version__)

## 2. Configuration (edit here)

In [ ]:
SOURCE   = "hf"        # "hf" (ImageNet-1k validation, needs HF token) | "imagenette" (indicative only) | "synthetic" (smoke test)
N_IMAGES = 600         # protocol minimum for P0 is 500
QUALITY  = 75
SKIP     = ["glass_blur"]   # breaks under current scikit-image
SEED     = 0
N_JOBS   = -1
PUSH_TO_GITHUB = False      # True: commit results (CSV/JSON only) to the repo
GITHUB_REPO        = "ArashSalehpourac/p68-hi-benefit-feasibility"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"    # name of the Colab secret holding your GitHub token
HF_SECRET_NAME     = "HF_TOKEN"        # name of the Colab secret holding your Hugging Face token
OUT_SUBDIR = "P68/outputs/p0"          # created under My Drive when running in Colab

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/{OUT_SUBDIR}"
else:
    OUT_DIR = os.path.abspath("p0_out")
os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", OUT_DIR)

## 3. Image source

In [ ]:
def secret(name):
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

def iter_hf(n):
    from datasets import load_dataset
    tok = secret(HF_SECRET_NAME)
    if not tok:
        raise RuntimeError(f"Add a Colab secret named {HF_SECRET_NAME} (and accept the ImageNet-1k terms on huggingface.co/datasets/ILSVRC/imagenet-1k).")
    ds = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True, token=tok)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    for i, ex in enumerate(ds):
        if i >= n: break
        yield f"inval_{i:05d}_c{ex['label']}", ex["image"].convert("RGB")

def iter_imagenette(n):
    import tarfile, urllib.request, glob
    root = "/content/imagenette" if IN_COLAB else os.path.join(OUT_DIR, "imagenette")
    if not os.path.isdir(os.path.join(root, "imagenette2-320")):
        os.makedirs(root, exist_ok=True)
        tgz = os.path.join(root, "imagenette2-320.tgz")
        urllib.request.urlretrieve("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz", tgz)
        tarfile.open(tgz).extractall(root)
    files = sorted(glob.glob(os.path.join(root, "imagenette2-320", "val", "*", "*.JPEG")))
    random.Random(SEED).shuffle(files)
    for p in files[:n]:
        yield os.path.basename(p), Image.open(p).convert("RGB")

def iter_synthetic(n):
    rng = np.random.default_rng(SEED)
    for i in range(n):
        h = w = 320
        base = rng.normal(128, 40, (h // 8, w // 8, 3))
        img = np.kron(base, np.ones((8, 8, 1))) + rng.normal(0, 6, (h, w, 3))
        yield f"syn_{i:04d}", Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))

LOADERS = {"hf": iter_hf, "imagenette": iter_imagenette, "synthetic": iter_synthetic}
images = list(LOADERS[SOURCE](N_IMAGES))
print(len(images), "images from", SOURCE)

## 4. Measure encoded size (decode → corrupt → encode once)

In [ ]:
def to_224(im):
    w, h = im.size
    s = 256 / min(w, h)
    im = im.resize((max(256, round(w * s)), max(256, round(h * s))), Image.BILINEAR)
    w, h = im.size
    l, t = (w - 224) // 2, (h - 224) // 2
    return np.asarray(im.crop((l, t, l + 224, t + 224)), dtype=np.uint8)

def enc_bytes(arr, fmt, q):
    buf = io.BytesIO(); im = Image.fromarray(arr)
    if fmt == "jpeg":
        im.save(buf, "JPEG", quality=q, optimize=False, progressive=False, subsampling="4:2:0")
    else:
        im.save(buf, "WEBP", quality=q, method=4)
    return buf.tell()

NAMES = [c for c in get_corruption_names() if c not in SKIP]

def one_image(idx, iid, im):
    import numpy as np
    if not hasattr(np, "float_"):              # joblib workers are fresh processes: re-apply the NumPy-2 shim here
        np.float_ = np.float64
    np.random.seed(SEED + idx)                 # reproducible corruption noise
    x = to_224(im); rows = []
    for fmt in ("jpeg", "webp"):
        rows.append((iid, "clean", 0, fmt, enc_bytes(x, fmt, QUALITY)))
    for c in NAMES:
        for s in range(1, 6):
            try:
                y = np.uint8(corrupt(x, corruption_name=c, severity=s))
            except Exception as e:
                rows.append((iid, c, s, "ERR", -1)); continue
            for fmt in ("jpeg", "webp"):
                rows.append((iid, c, s, fmt, enc_bytes(y, fmt, QUALITY)))
    return rows

CSV = os.path.join(OUT_DIR, "payload_p0.csv")
done = set()
if os.path.exists(CSV):
    done = set(pd.read_csv(CSV, usecols=["image"])["image"])
    print("resuming; already done:", len(done))
todo = [(i, iid, im) for i, (iid, im) in enumerate(images) if iid not in done]
t0 = time.time(); CHUNK = 40
for k in range(0, len(todo), CHUNK):
    chunk = todo[k:k + CHUNK]
    out = Parallel(n_jobs=N_JOBS)(delayed(one_image)(i, iid, im) for i, iid, im in chunk)
    df_new = pd.DataFrame([r for rows in out for r in rows], columns=["image", "corruption", "severity", "codec", "bytes"])
    df_new.to_csv(CSV, mode="a", header=not os.path.exists(CSV), index=False)
    print(f"{min(k + CHUNK, len(todo))}/{len(todo)} images | {time.time() - t0:.0f}s")
df = pd.read_csv(CSV)
nerr = int((df.codec == "ERR").sum())
print("errors:", nerr, "| rows:", len(df))
if nerr:
    print("WARNING: corruptions with failed rows ->", df[df.codec == "ERR"].groupby("corruption").size().to_dict(),
          "\nFix before interpreting results (a missing corruption silently shrinks the family table).")

## 5. Per-family size direction (pre-registered in v4-final §4)

Confirmatory families: **additive noise** (gaussian, shot, impulse) → payload expected to **increase**; **reductive** (defocus blur, contrast) → payload expected to **decrease**. (v4-final also listed "Gaussian blur", but it is not one of the 15 ImageNet-C corruptions; see the post-freeze erratum.) Everything else is exploratory (fog explicitly so). This checks only the *size direction*, a necessary precondition for the predicted sign of κ_mech; it says nothing about benefit G.

In [ ]:
FAMILY = {
    "gaussian_noise": ("confirmatory-additive", +1), "shot_noise": ("confirmatory-additive", +1), "impulse_noise": ("confirmatory-additive", +1),
    "defocus_blur": ("confirmatory-reductive", -1), "contrast": ("confirmatory-reductive", -1),
}
d = df[df.codec.isin(["jpeg", "webp"])].copy()
clean = d[d.corruption == "clean"].set_index(["image", "codec"])["bytes"].rename("clean_bytes")
d = d[d.corruption != "clean"].join(clean, on=["image", "codec"])
d["ratio"] = d["bytes"] / d["clean_bytes"]

rows = []
for (codec, c), g in d.groupby(["codec", "corruption"]):
    med = g.groupby("severity")["ratio"].median()
    if len(med) < 5: continue
    rho = spearmanr(g["severity"], g["ratio"]).correlation
    fam, pred = FAMILY.get(c, ("exploratory", 0))
    r5 = med.loc[5]
    direction = "up" if r5 > 1.02 else ("down" if r5 < 0.98 else "flat")
    ok = None if pred == 0 else ((pred > 0 and direction == "up") or (pred < 0 and direction == "down"))
    rows.append(dict(codec=codec, corruption=c, family=fam, sev1_ratio=med.loc[1], sev5_ratio=r5, rho=rho,
                     direction=direction, matches_prediction=ok,
                     clean_kB=g["clean_bytes"].median() / 1000, sev5_kB=g[g.severity == 5]["bytes"].median() / 1000))
summ = pd.DataFrame(rows).sort_values(["codec", "family", "corruption"])
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda v: f"{v:.3f}")
display(summ)
conf = summ[summ.family != "exploratory"]
print("\nconfirmatory rows matching prediction:", int(conf.matches_prediction.sum()), "/", len(conf))
print("mean clean payload (kB):", round(summ.clean_kB.mean(), 1), "(whole-image JPEG/WebP at 224x224, q%d)" % QUALITY)

## 6. Save results (CSV/JSON only; no images)

In [ ]:
summ.to_csv(os.path.join(OUT_DIR, "summary_p0.csv"), index=False)
import imagecorruptions, scipy, pandas
meta = dict(date=time.strftime("%Y-%m-%d %H:%M:%S"), source=SOURCE, n_images=len(images), quality=QUALITY, seed=SEED,
            skipped=SKIP, corruptions=NAMES, python=platform.python_version(), numpy=np.__version__, pandas=pandas.__version__,
            scipy=scipy.__version__, imagecorruptions=getattr(imagecorruptions, "__version__", "unknown"),
            image_ids_sha256=hashlib.sha256("\n".join(i for i, _ in images).encode()).hexdigest(),
            protocol="proposal_v4_final.md (pilot protocol frozen 2026-10-01)",
            evidence_note="SOURCE != 'hf' is NOT protocol evidence" if SOURCE != "hf" else "ImageNet-1k validation sample (streamed, shuffled)")
json.dump(meta, open(os.path.join(OUT_DIR, "run_meta_p0.json"), "w"), indent=2)
print(json.dumps(meta, indent=2)[:900])

## 7. (Optional) Commit results to GitHub

Set `PUSH_TO_GITHUB = True` above and add a Colab secret with your GitHub token (contents write access to the repo). Only `results/p0/*.csv|json` is committed. The token is never printed.

In [ ]:
if PUSH_TO_GITHUB:
    tok = secret(GITHUB_SECRET_NAME)
    assert tok, f"Add a Colab secret named {GITHUB_SECRET_NAME}"
    hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{tok}".encode()).decode()
    repo_dir = "/content/p68_repo" if IN_COLAB else os.path.join(OUT_DIR, "p68_repo")
    def git(*a, check=True):
        r = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader={hdr}", *a], cwd=None if a[0] == "clone" else repo_dir,
                           capture_output=True, text=True)
        if check and r.returncode: raise RuntimeError(r.stderr.replace(tok, "***"))
        return r.stdout
    if not os.path.isdir(repo_dir):
        git("clone", f"https://github.com/{GITHUB_REPO}.git", repo_dir)
    dst = os.path.join(repo_dir, "results", "p0"); os.makedirs(dst, exist_ok=True)
    for f in ("payload_p0.csv", "summary_p0.csv", "run_meta_p0.json"):
        subprocess.run(["cp", os.path.join(OUT_DIR, f), dst], check=True)
    git("config", "user.name", "Arash Salehpour"); git("config", "user.email", "arash.salehpour4@gmail.com")
    git("add", "results/p0"); git("commit", "-m", f"P0 payload run ({SOURCE}, n={len(images)})", check=False)
    print(git("push", "origin", "HEAD").strip() or "pushed")
else:
    print("PUSH_TO_GITHUB is False; results are only in", OUT_DIR)